# Practice 105 — Synthetic Control & Synthetic DiD

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import xy.pyplot as plt

from src.datasets import load_dataset, TREATMENT_YEAR
from src.plotting import gap_plot, placebo_spaghetti_plot, treated_vs_synthetic_plot

## Phase 0 — The data: California's Proposition 99

California passed Proposition 99 in November 1988, raising the cigarette tax
and funding a tobacco-control program. We have annual per-capita cigarette
sales for California and 38 other US states (the donor pool) from 1970-2000.
There is exactly one treated unit and no natural control — the setting this
whole practice is about.

In [ ]:
data = load_dataset()
print(f"Treated unit: California, treatment year: {TREATMENT_YEAR}")
print(f"Donor pool: {len(data.donor_names)} states")
print(f"Years: {data.years.min()}-{data.years.max()}")
pd.DataFrame({"California": data.y_treated}, index=data.years).head()

### Donor-pool selection

The donor pool here already excludes states that ran a large tobacco-control
program of their own during 1970-2000 (Massachusetts, Arizona, Oregon,
Florida and several others) — if we let those states into the pool, the
"control" units would themselves be partially treated, contaminating the
counterfactual. This exclusion is a substantive judgment call made *before*
any weight is fit — no amount of optimization can undo a badly chosen donor
pool. See `CLAUDE.md` "Theoretical Context" for the general version of this
problem.

## Phase 1 — Pre-treatment fit loss (RMSPE)

Before we can compare synthetic controls to each other (Phase 2) or rank
placebos against the true effect (Phase 4), we need one number that
summarizes how far apart two trajectories are: root-mean-squared prediction
error.

### Exercise — `src/_01_fit_loss.py :: rmspe`

Open `src/_01_fit_loss.py`, read the `TODO(human)` block above the function,
implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_fit_loss import rmspe

# Sanity check on a naive baseline (equal-weighted donor average) --
# not yet the optimized synthetic control, just enough to exercise rmspe().
y_naive = data.Y_donors.mean(axis=1)
naive_pre = rmspe(data.y_treated[data.pre_mask], y_naive[data.pre_mask])
print(f"Naive (equal-weight) pre-treatment RMSPE: {naive_pre:.3f}")

## Phase 2 — Simplex-constrained synthetic-control weights

The core estimator. We choose donor weights `w` — non-negative, summing to
one — that make the weighted donor average track California's
*pre-treatment* outcome path as closely as possible. This is a quadratic
program over the probability simplex: the constraint set is exactly what a
background in constrained optimization would recognize immediately, and it's
what buys synthetic control its two signature properties — no
extrapolation, and weights you can read off and interpret directly ("80%
Nevada, 12% Montana, ...").

### Exercise — `src/_02_synthetic_weights.py :: solve_synthetic_weights`

Open `src/_02_synthetic_weights.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._02_synthetic_weights import compare_to_pysyncon, fit_synthetic_control

fit = fit_synthetic_control(data)
top5 = sorted(zip(data.donor_names, fit.weights), key=lambda kv: -kv[1])[:5]
print(f"Pre-treatment RMSPE: {fit.pre_rmspe:.3f}")
print("Top 5 donor weights:")
for name, w in top5:
    print(f"  {name:16s} {w:.3f}")

fig = treated_vs_synthetic_plot(data.years, data.y_treated, fit.y_synthetic, TREATMENT_YEAR)
fig

### Validating against `pysyncon`

`pysyncon` implements the full Abadie, Diamond & Hainmueller method, nesting
our simplex QP (their "W" problem) inside an outer optimization over
predictor importance (their "V" problem). We configure it to skip that outer
loop — equal weight on every pre-treatment year, no aggregated covariates —
so it is solving (almost) the same objective we are. An exact weight-for-
weight match isn't expected (their row-scaling step differs from ours), but
pre-treatment fit quality should land in the same neighborhood.

In [ ]:
compare_to_pysyncon(data)

## Phase 3 — The gap and the post-treatment effect

With a synthetic California in hand, the estimated effect is just the
difference between the two trajectories — small pre-treatment (a fit
check), and the actual estimate post-treatment.

### Exercise — `src/_03_gap_effect.py :: compute_gap`

Open `src/_03_gap_effect.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._03_gap_effect import compute_gap

gap_result = compute_gap(data, fit.y_synthetic)
print(f"Post-treatment average effect: {gap_result.post_treatment_effect:.2f} packs/capita/year")

fig = gap_plot(data.years, gap_result.gap, TREATMENT_YEAR)
fig

## Phase 4 — Placebo (permutation) inference

One treated unit means no textbook standard error. Instead, we re-run the
exact same method on every donor, pretending each one was treated, and rank
California's post/pre RMSPE ratio against that donor-generated null
distribution.

### Exercise — `src/_04_placebo_inference.py :: permutation_p_value`

Open `src/_04_placebo_inference.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below. This cell re-fits
a synthetic control for every one of the ~38 donors, so it takes noticeably
longer than the earlier cells.

In [ ]:
from src._04_placebo_inference import run_placebo_test

placebo_result = run_placebo_test(data)
rank = round(placebo_result.p_value * (len(placebo_result.ratios) + 1))
print(f"California post/pre RMSPE ratio: {placebo_result.true_ratio:.2f}")
print(f"Rank among {len(placebo_result.ratios) + 1} units (1 = most extreme): {rank}")
print(f"Permutation p-value: {placebo_result.p_value:.3f}")

fig = placebo_spaghetti_plot(
    data.years, placebo_result.gaps, placebo_result.true_gap, TREATMENT_YEAR
)
fig

## Phase 5 — End-to-end run & synthetic DiD

Synthetic DiD (Arkhangelsky, Athey, Hirshberg, Imbens & Wager 2021) is the
reconciliation of synthetic control with the DiD tradition: it keeps
synthetic control's simplex-constrained *unit* weights (exactly what Phase 2
computes) but adds a second set of *time* weights, and replaces synthetic
control's "match the level" pre-treatment fit with DiD's "match the trend"
— the final estimate is a difference-in-differences on top of a weighted
(rather than simple) average of donors. See `CLAUDE.md`, "Theoretical
Context," for the full comparison; implementing it is future work beyond
this practice's scope.

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert np.isclose(fit.weights.sum(), 1.0, atol=1e-4), "weights must sum to 1"
assert np.all(fit.weights >= -1e-8), "weights must be non-negative"
assert fit.pre_rmspe < naive_pre, "the optimized synthetic control should out-fit the naive baseline"
assert gap_result.gap.shape == data.years.shape
assert 0.0 <= placebo_result.p_value <= 1.0
print("OK")